# Retriever RRF

This notebook loads multiple pre-computed dense retriever score JSON files, applies parameterized Reciprocal Rank Fusion (RRF) with custom model weights, and serializes the top-k fused candidate IDs into unified cache files.

Current Configurations:
- [x] qwen3-embedding-8b (0.6) + bge-m3 (0.4)

Cached configurations can be found in `../outputs/cache/ensemble_dense_scores`

In [1]:
# === IMPORT LIBRARIES ===
import json
import os
from collections import defaultdict
from pathlib import Path

In [ ]:
# === CONFIGURATION ===
CACHE_DIR = Path('../outputs/cache')
ENSEMBLE_CACHE_DIR = Path('../outputs/cache/ensemble_dense_scores')

K_CONSTANT = 60
TOP_N_OUTPUT = 15

MODELS_CONFIG = {
    'qwen3-embedding-8b': {
        'weight': 0.6,
        'train_file': CACHE_DIR / 'qwen3-embedding-8b_dense_scores_train.json',
        'test_file': CACHE_DIR / 'qwen3-embedding-8b_dense_scores_test.json'
    },
    'bge-m3': {
        'weight': 0.4,
        'train_file': CACHE_DIR / 'bge-m3_dense_scores_train.json',
        'test_file': CACHE_DIR / 'bge-m3_dense_scores_test.json'
    }
}

# 1. Create a dynamic string representing all models and weights
# Result: "qwen3-embedding-8b_w0.6_bge-m3_w0.4"
config_str = "_".join([f"{name}_w{cfg['weight']}" for name, cfg in MODELS_CONFIG.items()])

# 2. Inject the dynamic string into your output paths
OUTPUT_TRAIN = ENSEMBLE_CACHE_DIR / f"{config_str}_dense_scores_train.json"
OUTPUT_TEST  = ENSEMBLE_CACHE_DIR / f"{config_str}_dense_scores_test.json"

print(OUTPUT_TRAIN.name)
print(OUTPUT_TEST.name)

qwen3-embedding-8b_w0.6_bge-m3_w0.4_dense_scores_train.json
qwen3-embedding-8b_w0.6_bge-m3_w0.4_dense_scores_test.json


In [ ]:
def load_scores(filepath):
    if not filepath.exists():
        print(f"Warning: File {filepath} not found.")
        return {}
    with open(filepath, 'r', encoding='utf-8') as f:
        return json.load(f)

In [ ]:
def perform_rrf(models_config, split='train'):
    all_scores = {}
    
    # Load all files
    for model_name, config in models_config.items():
        filepath = config[f'{split}_file']
        all_scores[model_name] = load_scores(filepath)
        
    # Get all query IDs (assuming the first available model has all of them)
    query_ids = set()
    for scores in all_scores.values():
        query_ids.update(scores.keys())
        
    fused_results = {}
    
    for q_id in query_ids:
        rrf_scores = defaultdict(float)
        
        for model_name, config in models_config.items():
            if q_id not in all_scores[model_name]:
                continue
                
            ranked_candidates = all_scores[model_name][q_id]
            weight = config['weight']
            
            for rank, candidate_id in enumerate(ranked_candidates):
                # rank is 0-indexed, standard RRF uses 1-indexed
                rrf_scores[candidate_id] += weight / (K_CONSTANT + rank + 1)
                
        # Sort by RRF score descending
        sorted_candidates = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
        
        # Keep top-N
        top_candidates = [c[0] for c in sorted_candidates[:TOP_N_OUTPUT]]
        fused_results[q_id] = top_candidates
        
    return fused_results

In [ ]:
# === RUN RRF FOR TRAIN SET ===
print("Running RRF for Train set...")
train_fused = perform_rrf(MODELS_CONFIG, split='train')

if train_fused:
    with open(OUTPUT_TRAIN, 'w', encoding='utf-8') as f:
        json.dump(train_fused, f, indent=2)
    print(f"Saved {len(train_fused)} ensembled results to {OUTPUT_TRAIN}")

Running RRF for Train set...
Saved 2055 ensembled results to ../outputs/cache/qwen3-embedding-8b_w0.6_bge-m3_w0.4_dense_scores_train.json

Running RRF for Test set...
Saved 1798 ensembled results to ../outputs/cache/qwen3-embedding-8b_w0.6_bge-m3_w0.4_dense_scores_test.json


In [ ]:
# === RUN RRF FOR TEST SET ===
print("\nRunning RRF for Test set...")
test_fused = perform_rrf(MODELS_CONFIG, split='test')

if test_fused:
    with open(OUTPUT_TEST, 'w', encoding='utf-8') as f:
        json.dump(test_fused, f, indent=2)
    print(f"Saved {len(test_fused)} ensembled results to {OUTPUT_TEST}")